# Sofifa Scraping

In [1]:
import csv
import time
import random
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright
import sys
import asyncio
import os
from playwright_stealth import Stealth

# Force Windows to use the Proactor Event Loop for subprocess support
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# REPLACE THIS with your massive URL containing all 46 columns
BASE_URL = "https://sofifa.com/players?&showCol%5B%5D=pi&showCol%5B%5D=ae&showCol%5B%5D=by&showCol%5B%5D=hi&showCol%5B%5D=pf&showCol%5B%5D=oa&showCol%5B%5D=pt&showCol%5B%5D=bp&showCol%5B%5D=gu&showCol%5B%5D=vl&showCol%5B%5D=wg&showCol%5B%5D=ta&showCol%5B%5D=cr&showCol%5B%5D=fi&showCol%5B%5D=he&showCol%5B%5D=sh&showCol%5B%5D=vo&showCol%5B%5D=ts&showCol%5B%5D=dr&showCol%5B%5D=cu&showCol%5B%5D=fr&showCol%5B%5D=lo&showCol%5B%5D=bl&showCol%5B%5D=to&showCol%5B%5D=ac&showCol%5B%5D=sp&showCol%5B%5D=ag&showCol%5B%5D=tp&showCol%5B%5D=so&showCol%5B%5D=ju&showCol%5B%5D=st&showCol%5B%5D=sr&showCol%5B%5D=ln&showCol%5B%5D=te&showCol%5B%5D=vi&showCol%5B%5D=pe&showCol%5B%5D=td&showCol%5B%5D=ma&showCol%5B%5D=sa&showCol%5B%5D=sl&showCol%5B%5D=tg&showCol%5B%5D=gd&showCol%5B%5D=gh&showCol%5B%5D=gc&showCol%5B%5D=gp&showCol%5B%5D=gr"
CSV_FILENAME = "data/sofifa/newdata/sofifa_players.csv"
TOTAL_PLAYERS = 400000 
PLAYERS_PER_PAGE = 60

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        # SoFifa's first column (the avatar picture) has no text in the header.
        # We dynamically rename this header to "ID" for our CSV.
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    return headers

def extract_rows_from_html(soup, limit=None):
    player_data = []
    rows = soup.select("table tbody tr")
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker (ads have very few columns)
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION (Strictly Isolated)
                if 'col-name' in classes:
                    links = td.find_all("a", href=lambda h: h and "/player/" in h)
                    name_text = ""
                    for a in links:
                        # Find the hyperlink that actually has the text of the name, not the avatar image
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        # Fallback just in case
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/player/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER STATS (Including the real ID column)
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
                    
            player_data.append(row_values)
            
            if limit and len(player_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return player_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    os.makedirs(os.path.dirname(CSV_FILENAME), exist_ok=True)

    with sync_playwright() as p:
        # 1. Setup Persistent Profile Folder
        profile_path = os.path.join(os.getcwd(), "data", "sofifa_profile")
        os.makedirs(profile_path, exist_ok=True)
        
        # 2. Launch actual Chrome with saved cookies
        context = p.chromium.launch_persistent_context(
            user_data_dir=profile_path,
            channel="chrome", 
            headless=False,
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        
        # 3. Resource Blocker
        def block_heavy_resources(route):
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked for extreme speed).\n")

        # 4. Grab the default tab and apply Stealth
        page = context.pages[0]
        stealth = Stealth()
        stealth.apply_stealth_sync(page)
        
        print("Launching persistent stealth browser to solve Cloudflare challenge...")
        
        try:
            # 3. Now it is safe to navigate
            page.goto(f"{BASE_URL}&offset=0")
            page.wait_for_selector("table tbody tr", timeout=30000)
            
            # Dynamic wait
            page.wait_for_function(
                "() => document.querySelectorAll('table tbody tr td').length > 100",
                timeout=15000
            )
            print("Challenge passed! Table loaded.")
            
        except Exception as e:
            print(f"Failed to bypass Cloudflare. Error: {e}")
            context.close()
            return

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-PLAYER VALIDATION TEST ---")
            
            # # Reload page with Turbo Mode active
            # page.goto(f"{BASE_URL}&offset=0")
            # page.wait_for_selector("table tbody tr") 
            
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            players = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, player in enumerate(players):
                player_dict = dict(zip(columns, player))
                print(f"Player {i+1}: {player_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            # 1. Check if we have an existing file to resume from
            start_offset = 0
            if os.path.exists(CSV_FILENAME):
                with open(CSV_FILENAME, "r", encoding="utf-8") as f:
                    # Count lines minus 1 for the header
                    existing_rows = sum(1 for line in f) - 1
                    if existing_rows > 0:
                        # Round down to the nearest multiple of 60 to ensure clean pagination
                        start_offset = (existing_rows // PLAYERS_PER_PAGE) * PLAYERS_PER_PAGE

            print(f"--- STARTING PRODUCTION SCRAPE ---")
            if start_offset > 0:
                print(f"[Resume Mode] Found {existing_rows} existing players. Resuming from offset {start_offset}...\n")
            else:
                print("[New Run] No existing data found. Starting from scratch...\n")

                # Initialize fresh CSV file with headers
                page.goto(f"{BASE_URL}&offset=0")
                page.wait_for_selector("table tbody tr")
                soup = BeautifulSoup(page.content(), "html.parser")
                headers = extract_headers_from_html(soup)
                
                with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                    writer = csv.writer(file)
                    writer.writerow(headers)
                
            # 2. Master Loop (Starts at start_offset)
            with tqdm(total=TOTAL_PLAYERS, initial=start_offset, desc="Scraping SoFifa", unit=" players") as pbar:
                for offset in range(start_offset, TOTAL_PLAYERS, PLAYERS_PER_PAGE):
                    url = f"{BASE_URL}&offset={offset}"
                    
                    success = False
                    for attempt in range(3):
                        try:
                            # Only navigate if it's not the first load (or if we are resuming)
                            if offset != 0 or start_offset > 0 or attempt > 0:
                                page.goto(url)
                                page.wait_for_selector("table tbody tr", timeout=30000)
                                
                                # Dynamic wait: Checks if the whole table has populated
                                page.wait_for_function(
                                    "() => document.querySelectorAll('table tbody tr td').length > 100",
                                    timeout=15000
                                )
                                
                            soup = BeautifulSoup(page.content(), "html.parser")
                            players = extract_rows_from_html(soup)
                            
                            # Append strictly to CSV (mode="a")
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(players)
                            
                            pbar.update(len(players))
                            success = True
                            
                            if len(players) == 0:
                                print("\n[Notice] No more players found. Database exhausted.")
                                context.close()
                                return
                            break 
                            
                        except Exception as e:
                            print(f"\n[Error on offset {offset}]. Attempt {attempt + 1}/3.")
                            print(f"Details: {str(e)}")  # <--- This will tell us exactly what failed
                            time.sleep(5)
                    
                    if not success:
                        print(f"\n[Fatal] Failed to fetch offset {offset} after 3 attempts. Stopping script to prevent data gaps.")
                        break

                    # Polite delay
                    time.sleep(random.uniform(1.5, 3.0))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        # Safely shut down Chromium
        context.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_16916\2740390964.py:15: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_16916\2740390964.py:15: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [2]:
run_test()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Failed to bypass Cloudflare. Error: Page.goto: Target page, context or browser has been closed
Call log:
  - navigating to "https://sofifa.com/players?&showCol%5B%5D=pi&showCol%5B%5D=ae&showCol%5B%5D=by&showCol%5B%5D=hi&showCol%5B%5D=pf&showCol%5B%5D=oa&showCol%5B%5D=pt&showCol%5B%5D=bp&showCol%5B%5D=gu&showCol%5B%5D=vl&showCol%5B%5D=wg&showCol%5B%5D=ta&showCol%5B%5D=cr&showCol%5B%5D=fi&showCol%5B%5D=he&showCol%5B%5D=sh&showCol%5B%5D=vo&showCol%5B%5D=ts&showCol%5B%5D=dr&showCol%5B%5D=cu&showCol%5B%5D=fr&showCol%5B%5D=lo&showCol%5B%5D=bl&showCol%5B%5D=to&showCol%5B%5D=ac&showCol%5B%5D=sp&showCol%5B%5D=ag&showCol%5B%5D=tp&showCol%5B%5D=so&showCol%5B%5D=ju&showCol%5B%5D=st&showCol%5B%5D=sr&showCol%5B%5D=ln&showCol%5B%5D=te&showCol%5B%5D=vi&showCol%5B%5D=pe&showCol%5B%5D=td&showCol%5B%5D=ma&showCol%5B%5D=sa&showCol%5B%5D=sl&showCol%5B%5D=tg&showCol%5B%5D=gd&sh

In [3]:
run_production()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Failed to bypass Cloudflare. Error: Page.goto: net::ERR_ABORTED; maybe frame was detached?
Call log:
  - navigating to "https://sofifa.com/players?&showCol%5B%5D=pi&showCol%5B%5D=ae&showCol%5B%5D=by&showCol%5B%5D=hi&showCol%5B%5D=pf&showCol%5B%5D=oa&showCol%5B%5D=pt&showCol%5B%5D=bp&showCol%5B%5D=gu&showCol%5B%5D=vl&showCol%5B%5D=wg&showCol%5B%5D=ta&showCol%5B%5D=cr&showCol%5B%5D=fi&showCol%5B%5D=he&showCol%5B%5D=sh&showCol%5B%5D=vo&showCol%5B%5D=ts&showCol%5B%5D=dr&showCol%5B%5D=cu&showCol%5B%5D=fr&showCol%5B%5D=lo&showCol%5B%5D=bl&showCol%5B%5D=to&showCol%5B%5D=ac&showCol%5B%5D=sp&showCol%5B%5D=ag&showCol%5B%5D=tp&showCol%5B%5D=so&showCol%5B%5D=ju&showCol%5B%5D=st&showCol%5B%5D=sr&showCol%5B%5D=ln&showCol%5B%5D=te&showCol%5B%5D=vi&showCol%5B%5D=pe&showCol%5B%5D=td&showCol%5B%5D=ma&showCol%5B%5D=sa&showCol%5B%5D=sl&showCol%5B%5D=tg&showCol%5B%5D=gd&showCo

In [4]:
import csv
import time
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import asyncio
from playwright.sync_api import sync_playwright
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# Exact URL, no offset parameters needed for leagues
BASE_URL = "https://sofifa.com/leagues"
CSV_FILENAME = "data/sofifa/newdata/sofifa_raw_leagues.csv"

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers():
    return ["sofifa_id", "sofifa_name", "sofifa_country"]

def extract_rows_from_html(soup, limit=None):
    league_data = []
    rows = soup.select("table tbody tr")
    
    if limit:
        rows = rows[:limit]
        
    for row in tqdm(rows, desc="Parsing Leagues", unit=" league"):
        try:
            cols = row.find_all("td")
            
            # Bulletproof check
            if len(cols) < 3:
                continue
                
            # 1. ID & NAME EXTRACTION
            name_td = cols[1]
            link = name_td.find("a", href=lambda h: h and "/league/" in h)
            
            league_id = link['href'].split('/')[2] if link else ""
            league_name = link.get_text(strip=True) if link else name_td.get_text(strip=True)
            
            # 2. BULLETPROOF COUNTRY EXTRACTION
            # Scans the entire row for the nation link instead of guessing the column index
            country_name = "Unknown"
            nation_link = row.find("a", href=lambda h: h and "na=" in h)
            
            if nation_link:
                # 1st Priority: Extract from the flag image's title attribute
                img = nation_link.find("img")
                if img and img.has_attr("title"):
                    country_name = img["title"]
                # 2nd Priority: Extract from the anchor tag's title attribute
                elif nation_link.has_attr("title"):
                    country_name = nation_link["title"]
            
            league_data.append([league_id, league_name, country_name])
            
        except Exception as e:
            continue
            
    return league_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    with sync_playwright() as p:
        # headless=False is required to pass Cloudflare's bot detection
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        print("Launching browser to solve potential Cloudflare challenge...")
        page.goto(BASE_URL)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Table loaded successfully.")
        except Exception as e:
            print("Failed to load table in time. Please try again.")
            browser.close()
            return

        # === RESOURCE BLOCKER (TURBO MODE) ===
        def block_heavy_resources(route):
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked).\n")

        # Extract the page source once
        soup = BeautifulSoup(page.content(), "html.parser")
        headers = extract_headers()

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 5-LEAGUE VALIDATION TEST ---")
            leagues = extract_rows_from_html(soup, limit=5)

            print(f"\nSuccessfully Fetched! Extracted {len(headers)} Columns.")
            print("-" * 50)
            for i, league in enumerate(leagues):
                league_dict = dict(zip(headers, league))
                print(f"League {i+1}: {league_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            print("--- STARTING SINGLE-PAGE PRODUCTION SCRAPE ---")
            
            leagues = extract_rows_from_html(soup)
            
            # Write to CSV in one clean operation
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
                writer.writerows(leagues)

            print(f"\nScraping complete! {len(leagues)} leagues safely saved to {CSV_FILENAME}")

        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_16916\2741405667.py:9: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_16916\2741405667.py:9: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [ ]:
run_test()

Launching browser to solve potential Cloudflare challenge...


In [ ]:
run_production()

Launching browser to solve potential Cloudflare challenge...
Table loaded successfully.
Turbo mode activated (Images/CSS blocked).

--- STARTING SINGLE-PAGE PRODUCTION SCRAPE ---


Parsing Leagues: 100%|██████████| 52/52 [00:00<00:00, 8702.22 league/s]


Scraping complete! 52 leagues safely saved to data/sofifa/newdata/sofifa_raw_leagues.csv


In [ ]:
import os
import re
import pandas as pd
import unicodedata
from rapidfuzz import fuzz

# ==========================================
# 1. CONFIGURATION & SEMANTIC MAPPING
# ==========================================
PATHS = {
    "matched": "newdata/leagues/matched",
    "partial": "newdata/leagues/partial match",
    "unmatched": "newdata/leagues/no match"
}
for path in PATHS.values():
    os.makedirs(path, exist_ok=True)

# Semantic dictionary to resolve geographic database inconsistencies
COUNTRY_MAP = {
    'england': 'united kingdom',
    'scotland': 'united kingdom',
    'wales': 'united kingdom',
    'northern ireland': 'united kingdom',
    'usa': 'united states',
    'korea republic': 'south korea',
    'republic of ireland': 'ireland',
    'china pr': 'china',
    'turkiye': 'turkey' 
}

def normalize_country(country_str):
    """Strips accents from country names and applies semantic mapping."""
    if pd.isna(country_str): return ""
    
    c = str(country_str).lower()
    c = unicodedata.normalize('NFKD', c).encode('ASCII', 'ignore').decode('utf-8')
    c = c.strip()
    
    return COUNTRY_MAP.get(c, c)

def clean_league_name(name):
    """Strips accents, punctuation, and standardizes text for the math engine."""
    if pd.isna(name): return ""
    name = str(name).lower()
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name = re.sub(r'[^a-z0-9\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# ==========================================
# 2. LOAD & PREP DATASETS
# ==========================================
master_df = pd.read_csv('data/unified_tables/leagues/matched/final_combined_leagues.csv')
sofifa_df = pd.read_csv('data/sofifa/newdata/sofifa_raw_leagues.csv')
# Enforce the sofifa_ prefix for column clarity downstream
sofifa_df = sofifa_df.rename(columns=lambda x: x if x.startswith('sofifa_') else f'sofifa_{x}')

# Dimensionality Reduction (Squash time-series)
unique_base_leagues = master_df[['soccersolver_id', 'soccersolver_name', 'soccersolver_country']].drop_duplicates()

# ==========================================
# 3. TRIAGE MATCHING ENGINE (DUAL-SCORING)
# ==========================================
potential_matches = []

for _, base_row in unique_base_leagues.iterrows():
    best_score = 0
    best_match_id = None
    
    base_country_norm = normalize_country(base_row['soccersolver_country'])
    base_name_clean = clean_league_name(base_row['soccersolver_name'])
    
    for _, sof_row in sofifa_df.iterrows():
        sof_country_norm = normalize_country(sof_row['sofifa_country'])
        
        # Geographic Anchoring (Fast-Fail)
        if base_country_norm != sof_country_norm:
            continue
            
        sof_name_clean = clean_league_name(sof_row['sofifa_name'])
        
        # DUAL-SCORING ENGINE
        # Score 1: Token Sort (Handles inverted structures: "League Premier" vs "Premier League")
        score_token = fuzz.token_sort_ratio(base_name_clean, sof_name_clean)
        
        # Score 2: Stripped Ratio (Handles spacing inconsistencies: "La Liga 2" vs "LaLiga 2")
        score_stripped = fuzz.ratio(base_name_clean.replace(" ", ""), sof_name_clean.replace(" ", ""))
        
        # Take the absolute best score of the two methods
        score = max(score_token, score_stripped)
        
        if score > best_score:
            best_score = score
            best_match_id = sof_row['sofifa_id']
            
    # Capture everything 40% and above into the global pool
    if best_match_id and best_score >= 40:
        potential_matches.append({
            'soccersolver_id': base_row['soccersolver_id'],
            'sofifa_id': best_match_id,
            'sofifa_match_score': best_score
        })

pool_df = pd.DataFrame(potential_matches)

# ==========================================
# 4. GLOBAL DEDUPLICATION (1-TO-1 ENFORCER)
# ==========================================
if not pool_df.empty:
    pool_df = pool_df.sort_values('sofifa_match_score', ascending=False)
    pool_df = pool_df.drop_duplicates(subset=['soccersolver_id'])
    pool_df = pool_df.drop_duplicates(subset=['sofifa_id'])

# ==========================================
# 5. HISTORICAL EXPLOSION (PRE-ROUTING MERGE)
# ==========================================
# Merge the 1-to-1 dictionary onto the massive time-series master file
exploded_df = master_df.merge(pool_df, on='soccersolver_id', how='left')
# Pull in the rest of the Sofifa columns
final_df = exploded_df.merge(sofifa_df, on='sofifa_id', how='left')

# ==========================================
# 6. ZERO DATA LOSS ROUTING
# ==========================================
# 1. Perfect Matches (> 75)
perfect_df = final_df[final_df['sofifa_match_score'] > 75].copy()

# 2. Partial Matches (40 to 75)
partial_df = final_df[(final_df['sofifa_match_score'] >= 40) & (final_df['sofifa_match_score'] <= 75)].copy()
partial_df['APPROVED'] = '' 

# 3. SoccerSolver Orphans (Failed to hit 40% threshold)
base_orphans = final_df[final_df['sofifa_match_score'].isna()].copy()
sofifa_cols = [c for c in final_df.columns if c.startswith('sofifa_')]
base_orphans.drop(columns=sofifa_cols, errors='ignore', inplace=True)

# 4. Sofifa Orphans (Using index isolation)
matched_sofifa_ids = set(pool_df['sofifa_id'].dropna()) if not pool_df.empty else set()
sofifa_orphans = sofifa_df[~sofifa_df['sofifa_id'].isin(matched_sofifa_ids)].copy()

# ==========================================
# 7. EXPORTS & MATH VERIFICATION
# ==========================================
perfect_df.to_csv(os.path.join(PATHS['matched'], 'leagues_matched.csv'), index=False)
partial_df.to_csv(os.path.join(PATHS['partial'], 'leagues_partial.csv'), index=False)
base_orphans.to_csv(os.path.join(PATHS['unmatched'], 'soccersolver_leagues_unmatched.csv'), index=False)
sofifa_orphans.to_csv(os.path.join(PATHS['unmatched'], 'sofifa_leagues_unmatched.csv'), index=False)

print("\n--- Routing Complete ---")
print(f"- Perfect Matches (Rows): {len(perfect_df)}")
print(f"- Require Manual Review (Rows): {len(partial_df)}")
print(f"- Unmatched SoccerSolver (Rows): {len(base_orphans)}")
print(f"- Unmatched Sofifa (Entities): {len(sofifa_orphans)}")

total_ss_processed = len(perfect_df) + len(partial_df) + len(base_orphans)
print(f"\n[MATH CHECK] Perfect + Partial + Orphans = {total_ss_processed} (Should exactly equal {len(master_df)})")


--- Routing Complete ---
- Perfect Matches (Rows): 162
- Require Manual Review (Rows): 28
- Unmatched SoccerSolver (Rows): 77
- Unmatched Sofifa (Entities): 24

[MATH CHECK] Perfect + Partial + Orphans = 267 (Should exactly equal 267)


In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
PATH_MATCHED = 'newdata/leagues/matched/leagues_matched.csv'
PATH_PARTIAL = 'newdata/leagues/partial match/leagues_partial.csv' 
PATH_FINAL_OUTPUT = 'newdata/leagues/matched/master_leagues_final_integrated.csv'

# ==========================================
# 1. LOAD DATASETS
# ==========================================
try:
    perfect_df = pd.read_csv(PATH_MATCHED)
    partial_df = pd.read_csv(PATH_PARTIAL)
except FileNotFoundError as e:
    print(f"Error: {e}. Ensure you have run the matching script and reviewed the partials.")
    raise

# ==========================================
# 2. ISOLATE APPROVED PARTIALS
# ==========================================
# Safely convert the APPROVED column to numeric, ignoring text or blanks
partial_df['APPROVED'] = pd.to_numeric(partial_df['APPROVED'], errors='coerce')

# Isolate explicitly approved leagues
approved_partials = partial_df[partial_df['APPROVED'] == 1].copy()

# Drop the helper column to align schemas perfectly
approved_partials.drop(columns=['APPROVED'], errors='ignore', inplace=True)

# ==========================================
# 3. INTEGRATION & EXPORT
# ==========================================
# Concatenate vertically
final_master_df = pd.concat([perfect_df, approved_partials], ignore_index=True)

# Export the final master file
final_master_df.to_csv(PATH_FINAL_OUTPUT, index=False)

# ==========================================
# 4. VERIFICATION
# ==========================================
print("--- Post-Review Integration Complete ---")
print(f"Perfect Matches Loaded:    {len(perfect_df)}")
print(f"Approved Partials Added:   {len(approved_partials)}")
print(f"Total Leagues in Master:   {len(final_master_df)}")
print(f"\nFinal dataset saved to: {PATH_FINAL_OUTPUT}")

--- Post-Review Integration Complete ---
Perfect Matches Loaded:    162
Approved Partials Added:   7
Total Leagues in Master:   169

Final dataset saved to: newdata/leagues/matched/master_leagues_final_integrated.csv


In [ ]:
import csv
import sys
import time
import random
import asyncio
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION & URL HANDLING
# ==========================================
# REPLACE THIS with your massive URL containing all custom team columns
CUSTOM_URL = "https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps" 

# Season Controls (Leave empty for current season)
FIFA_VERSION = "" 
ROSTER_ID = ""    

CSV_FILENAME = "data/sofifa/newdata/teams-seasonwise/FC26.csv"
TOTAL_TEAMS = 1500
TEAMS_PER_PAGE = 60

# Safely inject season parameters into your custom URL
if FIFA_VERSION and ROSTER_ID:
    separator = "&" if "?" in CUSTOM_URL else "?"
    BASE_URL = f"{CUSTOM_URL}{separator}r={ROSTER_ID}&set={FIFA_VERSION}"
else:
    BASE_URL = CUSTOM_URL

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        
        # Dynamically rename the crest picture column to Team_ID
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("Team_ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    # Add a Season ID column header
    headers.append("Season_Version")
    return headers

def extract_rows_from_html(soup, limit=None):
    team_data = []
    rows = soup.select("table tbody tr")
    
    season_tag = FIFA_VERSION if FIFA_VERSION else "Latest"
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION
                if any(c in classes for c in ['col-name', 'col-name-wide']):
                    links = td.find_all("a", href=lambda h: h and "/team/" in h)
                    name_text = ""
                    for a in links:
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/team/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER CUSTOM STATS
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
            
            row_values.append(season_tag)
            team_data.append(row_values)
            
            if limit and len(team_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return team_data

# ==========================================
# PHASE 3: THE BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        # Safely handle pagination parameters
        separator = "&" if "?" in BASE_URL else "?"
        initial_url = f"{BASE_URL}{separator}offset=0"
        
        print("Launching browser to solve Cloudflare challenge...")
        page.goto(initial_url)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Challenge passed! Table loaded.")
        except Exception as e:
            print("Failed to bypass Cloudflare in time. Please try again.")
            browser.close()
            return

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-TEAM VALIDATION TEST ---")
            
            # Page is already loaded from the challenge step, extract directly
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            teams = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, team in enumerate(teams):
                team_dict = dict(zip(columns, team))
                print(f"Team {i+1}: {team_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            season_display = FIFA_VERSION if FIFA_VERSION else "Latest Version"
            print(f"--- STARTING PRODUCTION SCRAPE (Up to {TOTAL_TEAMS} Teams | Season: {season_display}) ---")
            
            # Extract headers from the already loaded initial page
            soup = BeautifulSoup(page.content(), "html.parser")
            headers = extract_headers_from_html(soup)
            
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
                
            # Master Loop
            with tqdm(total=TOTAL_TEAMS, desc=f"Scraping Season {season_display}", unit=" teams") as pbar:
                for offset in range(0, TOTAL_TEAMS, TEAMS_PER_PAGE):
                    url = f"{BASE_URL}{separator}offset={offset}"
                    
                    for attempt in range(3):
                        try:
                            # For the first loop (offset=0), this reloads the same page, 
                            # but ensures the loop logic stays consistent
                            page.goto(url)
                            page.wait_for_selector("table tbody tr", timeout=30000)
                            
                            soup = BeautifulSoup(page.content(), "html.parser")
                            teams = extract_rows_from_html(soup)
                            
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(teams)
                            
                            pbar.update(len(teams))
                            
                            if len(teams) == 0:
                                print("\n[Notice] No more teams found. End of database reached.")
                                browser.close()
                                return
                            break
                            
                        except Exception as e:
                            # This will print the actual technical reason it failed
                            print(f"\n[Scrape Error]: {str(e)}")
                            print("Retrying in 10 seconds...")
                            time.sleep(10)
                    
                    time.sleep(random.uniform(2.0, 3.5))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_team_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_team_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_8420\3601019842.py:12: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_8420\3601019842.py:12: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [ ]:
run_team_test()

Launching browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- RUNNING 10-TEAM VALIDATION TEST ---
Successfully Fetched! Extracted 12 Columns.
--------------------------------------------------
Team 1: {'Unknown': '', 'Name': 'Paris Saint-Germain Ligue 1', 'ID': '73', 'Overall': '85', 'Attack': '85', 'Midfield': '86', 'Defence': '86', 'Transfer budget': '€232.4M', 'Club worth': '€4B', 'Players': '27', 'Season_Version': 'Latest'}
Team 2: {'Unknown': '', 'Name': 'FC Barcelona La Liga', 'ID': '241', 'Overall': '85', 'Attack': '87', 'Midfield': '85', 'Defence': '83', 'Transfer budget': '€77.5M', 'Club worth': '€4.9B', 'Players': '29', 'Season_Version': 'Latest'}
Team 3: {'Unknown': '', 'Name': 'Real Madrid La Liga', 'ID': '243', 'Overall': '85', 'Attack': '90', 'Midfield': '84', 'Defence': '82', 'Transfer budget': '€146M', 'Club worth': '€5.8B', 'Players': '35', 'Season_Version': 'Latest'}
Team 4: {'Unknown': '', 'Name': 'Arsenal Premier League', 'ID': '1', 'Overall

In [ ]:
run_team_production()

Launching browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- STARTING PRODUCTION SCRAPE (Up to 1500 Teams | Season: Latest Version) ---


Scraping Season Latest Version:  29%|██▉       | 438/1500 [04:43<11:00,  1.61 teams/s]


[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible

Retrying in 10 seconds...


Scraping Season Latest Version:  32%|███▏      | 476/1500 [06:30<21:39,  1.27s/ teams]


[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish...
    - navigated to "https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720"

Retrying in 10 seconds...

[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish..

In [ ]:
# --- NEW RESUME VARIABLE ---
# 420 covers teams 421-480. We do this to ensure no dropped records from the crash.
START_OFFSET = 660  

if FIFA_VERSION and ROSTER_ID:
    separator = "&" if "?" in CUSTOM_URL else "?"
    BASE_URL = f"{CUSTOM_URL}{separator}r={ROSTER_ID}&set={FIFA_VERSION}"
else:
    BASE_URL = CUSTOM_URL

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("Team_ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    headers.append("Season_Version")
    return headers

def extract_rows_from_html(soup, limit=None):
    team_data = []
    rows = soup.select("table tbody tr")
    
    season_tag = FIFA_VERSION if FIFA_VERSION else "Latest"
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                if any(c in classes for c in ['col-name', 'col-name-wide']):
                    links = td.find_all("a", href=lambda h: h and "/team/" in h)
                    name_text = ""
                    for a in links:
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/team/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
            
            row_values.append(season_tag)
            team_data.append(row_values)
            
            if limit and len(team_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return team_data

# ==========================================
# PHASE 3: THE BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=False):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        separator = "&" if "?" in BASE_URL else "?"
        initial_url = f"{BASE_URL}{separator}offset={START_OFFSET}"
        
        print(f"Launching browser to solve Cloudflare challenge at offset {START_OFFSET}...")
        page.goto(initial_url)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Challenge passed! Table loaded.")
        except Exception as e:
            print(f"Failed to bypass Cloudflare. Error: {str(e)}")
            browser.close()
            return

        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        season_display = FIFA_VERSION if FIFA_VERSION else "Latest Version"
        print(f"--- RESUMING SCRAPE (From offset {START_OFFSET} up to {TOTAL_TEAMS} Teams | Season: {season_display}) ---")
        
        # If starting from 0, write headers. If resuming, skip header writing.
        if START_OFFSET == 0:
            soup = BeautifulSoup(page.content(), "html.parser")
            headers = extract_headers_from_html(soup)
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
        else:
            print("Appending to existing CSV to prevent data overwrite...")
            
        # Master Loop
        # tqdm's `initial` parameter visually sets the progress bar to where we left off
        with tqdm(total=TOTAL_TEAMS, initial=START_OFFSET, desc=f"Scraping Season {season_display}", unit=" teams") as pbar:
            for offset in range(START_OFFSET, TOTAL_TEAMS, TEAMS_PER_PAGE):
                url = f"{BASE_URL}{separator}offset={offset}"
                
                for attempt in range(3):
                    try:
                        page.goto(url)
                        page.wait_for_selector("table tbody tr", timeout=30000)
                        
                        soup = BeautifulSoup(page.content(), "html.parser")
                        teams = extract_rows_from_html(soup)
                        
                        # Mode is strictly "a" to append seamlessly
                        with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                            writer = csv.writer(file)
                            writer.writerows(teams)
                        
                        pbar.update(len(teams))
                        
                        if len(teams) == 0:
                            print("\n[Notice] No more teams found. End of database reached.")
                            browser.close()
                            return
                        break
                        
                    except Exception as e:
                        print(f"\n[Scrape Error]: {str(e)}")
                        print("Retrying in 10 seconds...")
                        time.sleep(10)
                
                time.sleep(random.uniform(3.0, 6.0))

        print(f"\nScraping complete! Data safely appended to {CSV_FILENAME}")
        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_team_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

# Execute the resume
run_team_production()

Launching browser to solve Cloudflare challenge at offset 660...
Challenge passed! Table loaded.
--- RESUMING SCRAPE (From offset 660 up to 1500 Teams | Season: Latest Version) ---
Appending to existing CSV to prevent data overwrite...


Scraping Season Latest Version:  44%|████▍     | 663/1500 [00:20<1:36:13,  6.90s/ teams]


[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish...
    - navigated to "https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720"

Retrying in 10 seconds...

[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish..

In [ ]:
import re
import sys
import asyncio
import pandas as pd

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION & PATHS
# ==========================================
TEAMS_CSV_PATH = "data/sofifa/newdata/teams-seasonwise/RAW/FC26.csv"
LEAGUES_CSV_PATH = "data/sofifa/newdata/sofifa_raw_leagues.csv"  
OUTPUT_CSV_PATH = "data/sofifa/newdata/teams-seasonwise/CLEANED/FC26_cleaned.csv"

df_teams = pd.read_csv(TEAMS_CSV_PATH)
df_leagues = pd.read_csv(LEAGUES_CSV_PATH)

print(f"Loaded {len(df_teams)} raw team records.")

# ==========================================
# PHASE 2: COLUMN CLEANING
# ==========================================
columns_to_keep = [col for col in df_teams.columns if 'unknown' not in col.lower() and 'unnamed' not in col.lower()]
df_teams = df_teams[columns_to_keep]
df_teams.columns = df_teams.columns.str.strip()

# ==========================================
# PHASE 3: DYNAMIC REGEX & FALLBACKS
# ==========================================
df_leagues['sofifa_name'] = df_leagues['sofifa_name'].astype(str).str.strip()
df_leagues['sofifa_country'] = df_leagues['sofifa_country'].astype(str).str.strip()

league_to_country_map = dict(zip(df_leagues['sofifa_name'].str.lower(), df_leagues['sofifa_country']))

# Add base defaults for the overlapping leagues
fallback_map = {
    "premier league": "England",
    "championship": "England",
    "la liga": "Spain",
    "serie a": "Italy",
    "bundesliga": "Germany",
    "2. bundesliga": "Germany",
    "ligue 1": "France",
    "pro league": "Saudi Arabia", # Default to Saudi Arabia
    "super league": "Switzerland", # Default to Switzerland
    "eredivisie": "Netherlands",
    "major league soccer": "USA"
}

# Force the base defaults to overwrite anything inherited from the raw leagues file
league_to_country_map.update(fallback_map)

league_list = set(df_leagues['sofifa_name'].tolist() + list(fallback_map.keys()))
league_list_sorted = sorted([str(l) for l in league_list], key=len, reverse=True)

escaped_leagues = [re.escape(league) for league in league_list_sorted]
league_pattern_group = "|".join(escaped_leagues)

regex_pattern = re.compile(rf"^(.+)\s({league_pattern_group})$", re.IGNORECASE)

# ==========================================
# PHASE 4: EXTRACTION, MAPPING & OVERRIDES
# ==========================================
def split_team_and_league(name_string):
    if pd.isna(name_string):
        return None, None
    match = regex_pattern.match(str(name_string).strip())
    if match:
        return match.group(1).strip(), match.group(2).strip().title()
    return name_string, "Unknown/Unmatched League"

extracted_data = df_teams['Name'].apply(lambda x: pd.Series(split_team_and_league(x)))
df_teams['Team_Name'] = extracted_data[0]
df_teams['League_Name'] = extracted_data[1]

# Map Base Country
df_teams['country'] = df_teams['League_Name'].str.lower().map(league_to_country_map).fillna("Unknown")

# --- THE OVERRIDE DICTIONARY ---
# We force these specific names into their correct countries regardless of what their league implies
AMBIGUOUS_TEAM_COUNTRIES = {
    # Ukraine (Premier League)
    'dynamo kyiv': 'Ukraine',
    'shakhtar donetsk': 'Ukraine',
    
    # Russia (Premier League - Futureproofing)
    'zenit': 'Russia',
    'cska moscow': 'Russia',
    'spartak moscow': 'Russia',
    'lokomotiv moscow': 'Russia',
    
    # Austria (Bundesliga)
    'lask linz': 'Austria',
    'sk rapid': 'Austria',
    'sk sturm graz': 'Austria',
    'fk austria wien': 'Austria',
    'wolfsberger ac': 'Austria',
    'sv ried': 'Austria',
    'sc rheindorf altach': 'Austria',
    'wsg tirol': 'Austria',
    'grazer ak 1902': 'Austria',
    'tsv hartberg': 'Austria',
    
    # Belgium (Pro League)
    'union saint-gilloise': 'Belgium',
    'club brugge kv': 'Belgium',
    'krc genk': 'Belgium',
    'rsc anderlecht': 'Belgium',
    'oud-heverlee leuven': 'Belgium',
    'fcv dender eh': 'Belgium',
    'cercle brugge ksv': 'Belgium',
    
    # Greece (Super League)
    'olympiacos fc': 'Greece',
    'aek athens': 'Greece',
    'paok': 'Greece',
    'panathinaikos fc': 'Greece',
    
    # China (Super League)
    'shanghai shenhua': 'China PR',
    'shandong taishan': 'China PR',
    'chengdu rongcheng': 'China PR',
    'beijing guoan': 'China PR',
    'zhejiang fc': 'China PR',
    'tianjin jinmen tiger': 'China PR',
    'dalian yingbo': 'China PR',
    'changchun yatai': 'China PR',
    'qingdao west coast fc': 'China PR',
    'henan fc': 'China PR',
    'wuhan three towns': 'China PR',
    'meizhou hakka': 'China PR',
    'yunnan yukun': 'China PR',
    'shenzhen peng city': 'China PR',
    'qingdao hainiu fc': 'China PR',
    
    # India (Super League)
    'mohun bagan super giant': 'India',
    'sc delhi': 'India',
    'kerala blasters fc': 'India',
    'mohammedan sc': 'India',
    'east bengal fc': 'India',
    'united tigers sc': 'India',
    'chennaiyin fc': 'India',
    'fc goa': 'India',
    'bengaluru fc': 'India',
    'mumbai city fc': 'India',
    'jamshedpur fc': 'India',
    'punjab fc': 'India',
    'odisha fc': 'India',
    'northeast united': 'India'
}

# Apply Override
team_lower = df_teams['Team_Name'].str.lower().str.strip()
df_teams['country'] = team_lower.map(AMBIGUOUS_TEAM_COUNTRIES).fillna(df_teams['country'])

# --- CLEANUP ---
if 'Name' in df_teams.columns:
    df_teams.drop(columns=['Name'], inplace=True)

all_cols = list(df_teams.columns)
identity_cols = ['Team_Name', 'League_Name', 'country']
ordered_cols = identity_cols + [col for col in all_cols if col not in identity_cols]
df_teams = df_teams[ordered_cols]
print(df_teams.head())

Loaded 711 raw team records.
             Team_Name     League_Name  country   ID  Overall  Attack  \
0  Paris Saint-Germain         Ligue 1   France   73       85      85   
1         FC Barcelona         La Liga    Spain  241       85      87   
2          Real Madrid         La Liga    Spain  243       85      90   
3              Arsenal  Premier League  England    1       84      84   
4            Liverpool  Premier League  England    9       84      87   

   Midfield  Defence Transfer budget Club worth Players Season_Version  
0        86     86.0         €232.4M        €4B      27         Latest  
1        85     83.0          €77.5M      €4.9B      29         Latest  
2        84     82.0           €146M      €5.8B      35         Latest  
3        85     85.0         €153.3M      €2.5B      23         Latest  
4        84     83.0         €124.4M      €4.7B      30         Latest  


C:\Users\ashwy\AppData\Local\Temp\ipykernel_36944\2159457308.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_36944\2159457308.py:7: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [ ]:
# ==========================================
# PHASE 5: EXPORT CLEANED DATA
# ==========================================
df_teams.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nCleaned dataset successfully saved to: {OUTPUT_CSV_PATH}")


Cleaned dataset successfully saved to: data/sofifa/newdata/teams-seasonwise/CLEANED/FC26_cleaned.csv


In [1]:
#FC25
import csv
import sys
import time
import random
import asyncio
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION & URL HANDLING
# ==========================================
# Removed the hardcoded r=260033&set=true from the URL so the season controls can inject cleanly
CUSTOM_URL = "https://sofifa.com/teams?type=club&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps" 

# Season Controls for FC 25
FIFA_VERSION = "20" 
# You can change 250001 to whichever specific roster update snapshot you want for the 24/25 season
ROSTER_ID = "200001"    

CSV_FILENAME = "data/sofifa/newdata/teams-seasonwise/RAW/FIFA 20.csv"
TOTAL_TEAMS = 750
TEAMS_PER_PAGE = 60

# Safely inject season parameters into your custom URL
if FIFA_VERSION and ROSTER_ID:
    separator = "&" if "?" in CUSTOM_URL else "?"
    BASE_URL = f"{CUSTOM_URL}{separator}r={ROSTER_ID}&set={FIFA_VERSION}"
else:
    BASE_URL = CUSTOM_URL

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        
        # Dynamically rename the crest picture column to Team_ID
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("Team_ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    # Add a Season ID column header
    headers.append("Season_Version")
    return headers

def extract_rows_from_html(soup, limit=None):
    team_data = []
    rows = soup.select("table tbody tr")
    
    season_tag = FIFA_VERSION if FIFA_VERSION else "Latest"
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION
                if any(c in classes for c in ['col-name', 'col-name-wide']):
                    links = td.find_all("a", href=lambda h: h and "/team/" in h)
                    name_text = ""
                    for a in links:
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/team/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER CUSTOM STATS
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
            
            row_values.append(season_tag)
            team_data.append(row_values)
            
            if limit and len(team_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return team_data

# ==========================================
# PHASE 3: THE BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        # Safely handle pagination parameters
        separator = "&" if "?" in BASE_URL else "?"
        initial_url = f"{BASE_URL}{separator}offset=0"
        
        print("Launching browser to solve Cloudflare challenge...")
        page.goto(initial_url)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Challenge passed! Table loaded.")
        except Exception as e:
            print("Failed to bypass Cloudflare in time. Please try again.")
            browser.close()
            return

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-TEAM VALIDATION TEST ---")
            
            # Page is already loaded from the challenge step, extract directly
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            teams = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, team in enumerate(teams):
                team_dict = dict(zip(columns, team))
                print(f"Team {i+1}: {team_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            season_display = FIFA_VERSION if FIFA_VERSION else "Latest Version"
            print(f"--- STARTING PRODUCTION SCRAPE (Up to {TOTAL_TEAMS} Teams | Season: {season_display}) ---")
            
            # Extract headers from the already loaded initial page
            soup = BeautifulSoup(page.content(), "html.parser")
            headers = extract_headers_from_html(soup)
            
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
                
            # Master Loop
            with tqdm(total=TOTAL_TEAMS, desc=f"Scraping Season {season_display}", unit=" teams") as pbar:
                for offset in range(0, TOTAL_TEAMS, TEAMS_PER_PAGE):
                    url = f"{BASE_URL}{separator}offset={offset}"
                    
                    for attempt in range(3):
                        try:
                            # For the first loop (offset=0), this reloads the same page, 
                            # but ensures the loop logic stays consistent
                            page.goto(url)
                            page.wait_for_selector("table tbody tr", timeout=30000)
                            
                            soup = BeautifulSoup(page.content(), "html.parser")
                            teams = extract_rows_from_html(soup)
                            
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(teams)
                            
                            pbar.update(len(teams))
                            
                            if len(teams) == 0:
                                print("\n[Notice] No more teams found. End of database reached.")
                                browser.close()
                                return
                            break
                            
                        except Exception as e:
                            # This will print the actual technical reason it failed
                            print(f"\n[Scrape Error]: {str(e)}")
                            print("Retrying in 10 seconds...")
                            time.sleep(10)
                    
                    time.sleep(random.uniform(2.0, 3.5))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_team_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_team_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()


C:\Users\ashwy\AppData\Local\Temp\ipykernel_516\1747680772.py:13: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_516\1747680772.py:13: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [2]:
run_team_test()

Launching browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- RUNNING 10-TEAM VALIDATION TEST ---
Successfully Fetched! Extracted 12 Columns.
--------------------------------------------------
Team 1: {'Unknown': '', 'Name': 'Paris Saint-Germain Ligue 1', 'ID': '73', 'Overall': '86', 'Attack': '89', 'Midfield': '83', 'Defence': '85', 'Transfer budget': '€160M', 'Club worth': '€2.2B', 'Players': '33', 'Season_Version': '22'}
Team 2: {'Unknown': '', 'Name': 'Manchester City Premier League', 'ID': '10', 'Overall': '85', 'Attack': '85', 'Midfield': '87', 'Defence': '85', 'Transfer budget': '€176M', 'Club worth': '€3.4B', 'Players': '33', 'Season_Version': '22'}
Team 3: {'Unknown': '', 'Name': 'Liverpool Premier League', 'ID': '9', 'Overall': '84', 'Attack': '86', 'Midfield': '83', 'Defence': '85', 'Transfer budget': '€95M', 'Club worth': '€3.5B', 'Players': '33', 'Season_Version': '22'}
Team 4: {'Unknown': '', 'Name': 'Atlético Madrid La Liga', 'ID': '240', 'Overal

In [ ]:
run_team_production()

Launching browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- STARTING PRODUCTION SCRAPE (Up to 750 Teams | Season: 20) ---


Scraping Season 20:  67%|██████▋   | 503/750 [05:28<03:38,  1.13 teams/s]


[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&r=200001&set=20&offset=720" navigation to finish...
    - navigated to "https://sofifa.com/teams?type=club&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&r=200001&set=20&offset=720"

Retrying in 10 seconds...

[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&r=200001&set=20&offset=720" navigation to finish...
    

In [2]:
import csv
import sys
import time
import random
import asyncio
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION & URL HANDLING
# ==========================================
# Custom URL with columns, stripped of hardcoded season parameters
CUSTOM_URL = "https://sofifa.com/teams?type=club&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps" 

# Season Controls for FC 25
FIFA_VERSION = "20" 
ROSTER_ID = "200001"    

CSV_FILENAME = "data/sofifa/newdata/teams-seasonwise/RAW/FIFA 20.csv"
TOTAL_TEAMS = 750
TEAMS_PER_PAGE = 60

# --- NEW RESUME VARIABLE ---
# 660 covers teams 661-720. 
START_OFFSET = 668

if FIFA_VERSION and ROSTER_ID:
    separator = "&" if "?" in CUSTOM_URL else "?"
    BASE_URL = f"{CUSTOM_URL}{separator}r={ROSTER_ID}&set={FIFA_VERSION}"
else:
    BASE_URL = CUSTOM_URL

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("Team_ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    headers.append("Season_Version")
    return headers

def extract_rows_from_html(soup, limit=None):
    team_data = []
    rows = soup.select("table tbody tr")
    
    season_tag = FIFA_VERSION if FIFA_VERSION else "Latest"
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                if any(c in classes for c in ['col-name', 'col-name-wide']):
                    links = td.find_all("a", href=lambda h: h and "/team/" in h)
                    name_text = ""
                    for a in links:
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/team/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
            
            row_values.append(season_tag)
            team_data.append(row_values)
            
            if limit and len(team_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return team_data

# ==========================================
# PHASE 3: THE BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=False):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        separator = "&" if "?" in BASE_URL else "?"
        initial_url = f"{BASE_URL}{separator}offset={START_OFFSET}"
        
        print(f"Launching browser to solve Cloudflare challenge at offset {START_OFFSET}...")
        page.goto(initial_url)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Challenge passed! Table loaded.")
        except Exception as e:
            print(f"Failed to bypass Cloudflare. Error: {str(e)}")
            browser.close()
            return

        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        season_display = FIFA_VERSION if FIFA_VERSION else "Latest Version"
        print(f"--- RESUMING SCRAPE (From offset {START_OFFSET} up to {TOTAL_TEAMS} Teams | Season: {season_display}) ---")
        
        # If starting from 0, write headers. If resuming, skip header writing.
        if START_OFFSET == 0:
            soup = BeautifulSoup(page.content(), "html.parser")
            headers = extract_headers_from_html(soup)
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
        else:
            print("Appending to existing CSV to prevent data overwrite...")
            
        # Master Loop
        # tqdm's `initial` parameter visually sets the progress bar to where we left off
        with tqdm(total=TOTAL_TEAMS, initial=START_OFFSET, desc=f"Scraping Season {season_display}", unit=" teams") as pbar:
            for offset in range(START_OFFSET, TOTAL_TEAMS, TEAMS_PER_PAGE):
                url = f"{BASE_URL}{separator}offset={offset}"
                
                for attempt in range(3):
                    try:
                        page.goto(url)
                        page.wait_for_selector("table tbody tr", timeout=30000)
                        
                        soup = BeautifulSoup(page.content(), "html.parser")
                        teams = extract_rows_from_html(soup)
                        
                        # Mode is strictly "a" to append seamlessly
                        with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                            writer = csv.writer(file)
                            writer.writerows(teams)
                        
                        pbar.update(len(teams))
                        
                        if len(teams) == 0:
                            print("\n[Notice] No more teams found. End of database reached.")
                            browser.close()
                            return
                        break
                        
                    except Exception as e:
                        print(f"\n[Scrape Error]: {str(e)}")
                        print("Retrying in 10 seconds...")
                        time.sleep(10)
                
                time.sleep(random.uniform(3.0, 6.0))

        print(f"\nScraping complete! Data safely appended to {CSV_FILENAME}")
        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_team_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

# Execute the resume
run_team_production()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_24920\1472638986.py:12: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_24920\1472638986.py:12: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


Launching browser to solve Cloudflare challenge at offset 668...
Failed to bypass Cloudflare. Error: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&r=200001&set=20&offset=668" navigation to finish...
    - navigated to "https://sofifa.com/teams?type=club&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&r=200001&set=20&offset=668"



In [3]:
import os
import re
import sys
import glob
import asyncio
import pandas as pd

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION & PATHS
# ==========================================
RAW_DIR = "data/sofifa/newdata/teams-seasonwise/RAW/"
CLEANED_DIR = "data/sofifa/newdata/teams-seasonwise/CLEANED/"
LEAGUES_CSV_PATH = "data/sofifa/newdata/sofifa_raw_leagues.csv"

# Ensure output directory exists
os.makedirs(CLEANED_DIR, exist_ok=True)

# ==========================================
# PHASE 2: GLOBAL INITIALIZATION (RUN ONCE)
# ==========================================
print("Initializing Master Dictionaries and Regex Engine...")

df_leagues = pd.read_csv(LEAGUES_CSV_PATH)
df_leagues['sofifa_name'] = df_leagues['sofifa_name'].astype(str).str.strip()
df_leagues['sofifa_country'] = df_leagues['sofifa_country'].astype(str).str.strip()


Initializing Master Dictionaries and Regex Engine...


C:\Users\ashwy\AppData\Local\Temp\ipykernel_24920\4084118400.py:9: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_24920\4084118400.py:9: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [4]:

# Base Map from Sofifa Raw
league_to_country_map = dict(zip(df_leagues['sofifa_name'].str.lower(), df_leagues['sofifa_country']))

# The Fallback Map for overlapping league names
fallback_map = {
    "premier league": "England",
    "championship": "England",
    "la liga": "Spain",
    "serie a": "Italy",
    "serie b": "Italy",
    "bundesliga": "Germany",
    "2. bundesliga": "Germany",
    "ligue 1": "France",
    "pro league": "Saudi Arabia", 
    "super league": "Switzerland", 
    "eredivisie": "Netherlands",
    "major league soccer": "United States",
    "superliga": "Denmark"
}
league_to_country_map.update(fallback_map)

# Compile Regex
league_list = set(df_leagues['sofifa_name'].tolist() + list(fallback_map.keys()))
league_list_sorted = sorted([str(l) for l in league_list], key=len, reverse=True)
escaped_leagues = [re.escape(league) for league in league_list_sorted]
league_pattern_group = "|".join(escaped_leagues)
regex_pattern = re.compile(rf"^(.+)\s({league_pattern_group})$", re.IGNORECASE)

# --- THE HISTORICAL OVERRIDE DICTIONARY ---
# Keys must be strictly lowercase for mapping
AMBIGUOUS_TEAM_COUNTRIES = {
    # Ukraine (Premier League)
    'dynamo kyiv': 'Ukraine',
    'shakhtar donetsk': 'Ukraine',
    
    # Russia (Premier League)
    'zenit': 'Russia',
    'cska moscow': 'Russia',
    'spartak moscow': 'Russia',
    'lokomotiv moscow': 'Russia',
    'pfc cska': 'Russia',
    'fc lokomotiv': 'Russia',
    
    # Austria (Bundesliga)
    'lask linz': 'Austria',
    'sk rapid': 'Austria',
    'sk sturm graz': 'Austria',
    'fk austria wien': 'Austria',
    'wolfsberger ac': 'Austria',
    'sv ried': 'Austria',
    'sc rheindorf altach': 'Austria',
    'wsg tirol': 'Austria',
    'grazer ak 1902': 'Austria',
    'tsv hartberg': 'Austria',
    'admira wacker': 'Austria',
    'austria lustenau': 'Austria',
    'blau-weiß linz': 'Austria',
    'skn st. pölten': 'Austria',
    'sk austria klagenfurt': 'Austria',
    
    # Belgium (Pro League)
    'union saint-gilloise': 'Belgium',
    'club brugge kv': 'Belgium',
    'krc genk': 'Belgium',
    'rsc anderlecht': 'Belgium',
    'oud-heverlee leuven': 'Belgium',
    'fcv dender eh': 'Belgium',
    'cercle brugge ksv': 'Belgium',
    'rfc seraing': 'Belgium',
    'royal excel mouscron': 'Belgium',
    'sv zulte waregem': 'Belgium',
    'k beerschot va': 'Belgium',
    'waasland-beveren': 'Belgium',
    'royal antwerp fc': 'Belgium',
    'standard de liège': 'Belgium',
    'kaa gent': 'Belgium',
    'kv mechelen': 'Belgium',
    'sint-truidense vv': 'Belgium',
    'kv kortrijk': 'Belgium',
    'kvc westerlo': 'Belgium',
    'kv oostende': 'Belgium',
    'as eupen': 'Belgium',
    'rwd molenbeek': 'Belgium',
    'royal charleroi sporting club': 'Belgium',
    
    # Ecuador (Serie A)
    'ldu quito': 'Ecuador',
    'barcelona de guayaquil': 'Ecuador',
    'emelec': 'Ecuador',
    'independiente del valle': 'Ecuador',
    'delfín sc': 'Ecuador',
    '9 de octubre': 'Ecuador',
    'deportivo cuenca': 'Ecuador',
    'el nacional': 'Ecuador',
    'guayaquil city': 'Ecuador',
    'macará': 'Ecuador',
    'mushuc runa': 'Ecuador',
    'sd aucas': 'Ecuador',
    'técnico universitario': 'Ecuador',
    'universidad católica del ecuador': 'Ecuador',
    
    # Greece (Super League)
    'olympiacos fc': 'Greece',
    'aek athens': 'Greece',
    'paok': 'Greece',
    'panathinaikos fc': 'Greece',
    
    # China (Super League)
    'shanghai shenhua': 'China PR',
    'shandong taishan': 'China PR',
    'chengdu rongcheng': 'China PR',
    'beijing guoan': 'China PR',
    'zhejiang fc': 'China PR',
    'tianjin jinmen tiger': 'China PR',
    'dalian yingbo': 'China PR',
    'changchun yatai': 'China PR',
    'qingdao west coast fc': 'China PR',
    'henan fc': 'China PR',
    'wuhan three towns': 'China PR',
    'meizhou hakka': 'China PR',
    'yunnan yukun': 'China PR',
    'shenzhen peng city': 'China PR',
    'qingdao hainiu fc': 'China PR',
    'guangzhou fc': 'China PR',
    'hebei fc': 'China PR',
    'chongqing dangdai lifan': 'China PR',
    'dalian professional': 'China PR',
    'tianjin tianhai fc': 'China PR',
    'shanghai port': 'China PR',
    'guangzhou city': 'China PR',
    'shenzhen fc': 'China PR',
    'wuhan yangtze river fc': 'China PR',
    'cangzhou mighty lions fc': 'China PR',
    'qingdao fc': 'China PR',
    'nantong zhiyun': 'China PR',
    'beijing renhe': 'China PR',
    
    # India (Super League)
    'mohun bagan super giant': 'India',
    'sc delhi': 'India',
    'kerala blasters fc': 'India',
    'mohammedan sc': 'India',
    'east bengal fc': 'India',
    'united tigers sc': 'India',
    'chennaiyin fc': 'India',
    'fc goa': 'India',
    'bengaluru fc': 'India',
    'mumbai city fc': 'India',
    'jamshedpur fc': 'India',
    'punjab fc': 'India',
    'odisha fc': 'India',
    'northeast united': 'India'
}


In [5]:

# ==========================================
# PHASE 3: THE CORE CLEANING FUNCTION
# ==========================================
def split_team_and_league(name_string):
    if pd.isna(name_string):
        return None, None
    match = regex_pattern.match(str(name_string).strip())
    if match:
        return match.group(1).strip(), match.group(2).strip().title()
    return name_string, "Unknown/Unmatched League"

def clean_season_csv(input_path, output_path):
    print(f"\nProcessing: {os.path.basename(input_path)}")
    df = pd.read_csv(input_path)
    
    # Drop Unknown/Unnamed noise
    columns_to_keep = [col for col in df.columns if 'unknown' not in col.lower() and 'unnamed' not in col.lower()]
    df = df[columns_to_keep]
    df.columns = df.columns.str.strip()
    
    if 'Name' not in df.columns:
        print(" -> Error: 'Name' column not found. Skipping file.")
        return

    # Extract Name & League
    extracted_data = df['Name'].apply(lambda x: pd.Series(split_team_and_league(x)))
    df['Team_Name'] = extracted_data[0]
    df['League_Name'] = extracted_data[1]

    # Map Base Country
    df['country'] = df['League_Name'].str.lower().map(league_to_country_map).fillna("Unknown")

    # Apply Historical Overrides
    team_lower = df['Team_Name'].str.lower().str.strip()
    df['country'] = team_lower.map(AMBIGUOUS_TEAM_COUNTRIES).fillna(df['country'])

    # Cleanup and reorder
    df.drop(columns=['Name'], inplace=True)
    all_cols = list(df.columns)
    identity_cols = ['Team_Name', 'League_Name', 'country']
    ordered_cols = identity_cols + [col for col in all_cols if col not in identity_cols]
    df = df[ordered_cols]
    
    # Export
    df.to_csv(output_path, index=False)
    print(f" -> Success! Cleaned file saved to: {os.path.basename(output_path)}")

# ==========================================
# PHASE 4: BATCH EXECUTION LOOP
# ==========================================
# Grab all CSVs in the RAW directory
raw_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))

if not raw_files:
    print(f"No CSV files found in {RAW_DIR}")
else:
    for file_path in raw_files:
        filename = os.path.basename(file_path)
        
        # Explicitly skip FC26.csv since it is already cleaned
        if "FC26" in filename:
            print(f"\nSkipping {filename} (Already cleaned).")
            continue
            
        # Construct output filename (e.g., FC25.csv -> FC25_cleaned.csv)
        name, ext = os.path.splitext(filename)
        output_filename = f"{name}_cleaned{ext}"
        output_path = os.path.join(CLEANED_DIR, output_filename)
        
        # Execute Pipeline
        clean_season_csv(file_path, output_path)

print("\nBatch cleaning complete.")


Processing: FC24.csv
 -> Success! Cleaned file saved to: FC24_cleaned.csv

Processing: FC25.csv
 -> Success! Cleaned file saved to: FC25_cleaned.csv

Skipping FC26.csv (Already cleaned).

Processing: FIFA 20.csv
 -> Success! Cleaned file saved to: FIFA 20_cleaned.csv

Processing: FIFA 21.csv
 -> Success! Cleaned file saved to: FIFA 21_cleaned.csv

Processing: FIFA 22.csv
 -> Success! Cleaned file saved to: FIFA 22_cleaned.csv

Processing: FIFA 23.csv
 -> Success! Cleaned file saved to: FIFA 23_cleaned.csv

Batch cleaning complete.


In [7]:
import pandas as pd
import glob
import os

# Define the folder path where your 7 files are stored
sofifa_cleaned = 'data/sofifa/newdata/teams-seasonwise/CLEANED/'

# Grab all CSV files in that directory
sofifa_files = glob.glob(os.path.join(sofifa_cleaned, '*.csv'))

dataframes_list = []

for file in sofifa_files:
    df = pd.read_csv(file)
    filename = os.path.basename(file)
    
    # Target the FC26 file to fix the string anomaly
    if filename == 'FC26_cleaned.csv':
        df['Season_Version'] = df['Season_Version'].replace('Latest', 26.0)
        
    # Force the column to a uniform float type across every file
    df['Season_Version'] = df['Season_Version'].astype(float)
    
    dataframes_list.append(df)

print("Concatenating files...")

# Stack all 7 files vertically
sofifa_master = pd.concat(dataframes_list, ignore_index=True)

# Audit check: Print unique values to ensure 7 distinct, clean numbers
print("\nAudit - Unique Seasons in your new Master File:")
print(sofifa_master['Season_Version'].unique())

# Export the final result
output_filename = 'data/sofifa/newdata/SOFIFA_TEAMS.csv'
sofifa_master.to_csv(output_filename, index=False)

print(f"\nSuccess! Master file saved as '{output_filename}'.")

Concatenating files...

Audit - Unique Seasons in your new Master File:
[24. nan 25. 26. 20. 21. 22. 23.]

Success! Master file saved as 'data/sofifa/newdata/SOFIFA_TEAMS.csv'.


## Processing Teams dataset for matching

In [1]:
import pandas as pd
import string
import re
import unicodedata
from IPython.display import display, HTML

# Load data
df = pd.read_csv('data/sofifa/newdata/SOFIFA_TEAMS.csv')
initial_rows = len(df)

# Step 1: Namespace Isolation & Deduplication
df = df.add_prefix('sofifa_')
df = df.drop_duplicates()
deduped_rows = len(df)
exact_dupes_removed = initial_rows - deduped_rows

# Step 2: String Normalization Engine
def normalize_string(text):
    if pd.isna(text):
        return text
    text = str(text).lower()
    # Strip accents/diacritics
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['sofifa_clean_name'] = df['sofifa_Team_Name'].apply(normalize_string)

# Step 3: The Geographic Override
# 3A: Fix USA mapping
df.loc[df['sofifa_country'] == 'USA', 'sofifa_country'] = 'United States'

# 3B: Fix Canadian MLS Teams currently mapped to United States
canadian_mls_teams = ['toronto fc', 'vancouver whitecaps fc', 'cf montreal', 'montreal impact']
df.loc[df['sofifa_clean_name'].isin(canadian_mls_teams), 'sofifa_country'] = 'Canada'

# 3C: Fix Ecuadorian Teams currently trapped in Serie A Italy
ecuadorian_teams = ['deportivo cuenca', 'independiente del valle', 'macara', 'emelec', 'ldu quito']
df.loc[df['sofifa_clean_name'].isin(ecuadorian_teams), 'sofifa_country'] = 'Ecuador'

# Step 4: Robust Alias Hardcoding
alias_mapping = {
    'inter': 'inter milan',
    'ajax': 'afc ajax',
    'psv': 'psv eindhoven',
    'roma': 'as roma',
    'lazio': 'ss lazio',
    'napoli': 'ssc napoli',
    'celtic': 'celtic fc',
    'tigre': 'club atletico tigre',
    'colon': 'club atletico colon',
    'spurs': 'tottenham hotspur',
    'wolves': 'wolverhampton wanderers',
    'boca juniors': 'ca boca juniors', 
    'river plate': 'ca river plate'   
}
df['sofifa_clean_name'] = df['sofifa_clean_name'].replace(alias_mapping)

# Save Phase 1 Output
output_file = 'data/sofifa/newdata/SOFIFA_TEAMS_CLEANED_PHASE1.csv'
df.to_csv(output_file, index=False)


# =====================================================================
# --- JUPYTER-FRIENDLY VERIFICATION PROTOCOL ---
# =====================================================================

display(HTML("<h3>--- PHASE 1 VERIFICATION PROTOCOL ---</h3>"))

# 1. Row Count Check
row_check_data = {
    'Metric': ['Initial Rows', 'Exact Duplicates Removed', 'Final Rows'],
    'Value': [initial_rows, exact_dupes_removed, deduped_rows],
    'Status': ['-', 'PASS' if exact_dupes_removed == 1075 else 'FAIL', '-']
}
df_row_check = pd.DataFrame(row_check_data)

# 2. Geographic Integrity Check
usa_count = len(df[df['sofifa_country'] == 'USA'])

ecuador_check = df[df['sofifa_clean_name'].isin(ecuadorian_teams)]['sofifa_country'].value_counts().to_dict()
ecuador_pass = all(v == 'Ecuador' for v in ecuador_check.keys()) if ecuador_check else False

canada_check = df[df['sofifa_clean_name'].isin(canadian_mls_teams)]['sofifa_country'].value_counts().to_dict()
canada_pass = all(v == 'Canada' for v in canada_check.keys()) if canada_check else False

geo_check_data = {
    'Test': ["'USA' count (Expected: 0)", "Ecuadorian Teams Mapping", "Canadian MLS Teams Mapping"],
    'Result': [usa_count, str(ecuador_check), str(canada_check)],
    'Status': ['PASS' if usa_count == 0 else 'FAIL', 
               'PASS' if ecuador_pass else 'FAIL',
               'PASS' if canada_pass else 'FAIL']
}
df_geo_check = pd.DataFrame(geo_check_data)

# 3. Alias & Normalization Integrity Check
inter_count = len(df[df['sofifa_clean_name'] == 'inter'])
inter_milan_count = len(df[df['sofifa_clean_name'] == 'inter milan'])
ajax_count = len(df[df['sofifa_clean_name'] == 'afc ajax'])

alias_check_data = {
    'Test': ["'inter' count (Expected: 0)", "'inter milan' count (Expected > 0)", "'afc ajax' mapped correctly"],
    'Result': [inter_count, inter_milan_count, "Present" if ajax_count > 0 else "Missing"],
    'Status': ['PASS' if inter_count == 0 else 'FAIL', 
               'PASS' if inter_milan_count > 0 else 'FAIL',
               'PASS' if ajax_count > 0 else 'FAIL']
}
df_alias_check = pd.DataFrame(alias_check_data)

# Display Tables in Jupyter
display(HTML("<b>1. Row Count Check</b>"))
display(df_row_check.style.hide(axis="index"))

display(HTML("<br><b>2. Geographic Integrity Check</b>"))
display(df_geo_check.style.hide(axis="index"))

display(HTML("<br><b>3. Alias Integrity Check</b>"))
display(df_alias_check.style.hide(axis="index"))

Metric,Value,Status
Initial Rows,4910,-
Exact Duplicates Removed,1075,PASS
Final Rows,3835,-


Test,Result,Status
'USA' count (Expected: 0),0,PASS
Ecuadorian Teams Mapping,{'Ecuador': 16},PASS
Canadian MLS Teams Mapping,{'Canada': 15},PASS


Test,Result,Status
'inter' count (Expected: 0),0,PASS
'inter milan' count (Expected > 0),7,PASS
'afc ajax' mapped correctly,Present,PASS


In [6]:
import pandas as pd
import string
import re
import unicodedata
from thefuzz import fuzz
from IPython.display import display, HTML

# ==========================================
# 1. SETUP & DATA PREPARATION
# ==========================================

SOFIFA_FILE = 'data/sofifa/newdata/SOFIFA_TEAMS_CLEANED_PHASE1.csv'
SSWY_FILE = 'data/unified_tables/teams/matched/final_combined_teams.csv' 

df_sofifa = pd.read_csv(SOFIFA_FILE)
df_sswy = pd.read_csv(SSWY_FILE)

# Align SSWY America to match Sofifa America
df_sswy.loc[df_sswy['soccersolver_country'] == 'USA', 'soccersolver_country'] = 'United States'

# Map Sofifa float seasons to SSWY string formats
season_mapping = {
    26.0: '2025-2026',
    25.0: '2024-2025',
    24.0: '2023-2024',
    23.0: '2022-2023',
    22.0: '2021-2022',
    21.0: '2020-2021',
    20.0: '2019-2020'
}
df_sofifa['sofifa_Season_Version'] = df_sofifa['sofifa_Season_Version'].map(season_mapping)
# Assign temporary unique IDs to track records through the triage pipeline
df_sofifa['sofifa_uid'] = df_sofifa.index.astype(str) + "_S"
df_sswy['sswy_uid'] = df_sswy.index.astype(str) + "_T"

total_sofifa_initial = len(df_sofifa)
total_sswy_initial = len(df_sswy)

def normalize_string(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the exact same normalization engine to the target database
df_sswy['sswy_clean_ss_name'] = df_sswy['soccersolver_name'].apply(normalize_string)
df_sswy['sswy_clean_wy_name'] = df_sswy['wyscout_name'].apply(normalize_string)

# Scoring function
def calculate_fuzz(row):
    score_ss = fuzz.token_set_ratio(row['sofifa_clean_name'], row['sswy_clean_ss_name'])
    score_wy = fuzz.token_set_ratio(row['sofifa_clean_name'], row['sswy_clean_wy_name']) if pd.notna(row['sswy_clean_wy_name']) and row['sswy_clean_wy_name'] != "" else 0
    return max(score_ss, score_wy)

# ==========================================
# 2. THE MULTI-PASS MATCHER
# ==========================================

# --- PASS 1: Strict Geographic Exact Match ---
# Block by Country and Season
pass1_pool = pd.merge(
    df_sofifa, df_sswy, 
    left_on=['sofifa_country', 'sofifa_Season_Version'], 
    right_on=['soccersolver_country', 'soccersolver_season']
)

# Exact match condition on either SS or WY name
exact_mask = (pass1_pool['sofifa_clean_name'] == pass1_pool['sswy_clean_ss_name']) | \
             (pass1_pool['sofifa_clean_name'] == pass1_pool['sswy_clean_wy_name'])

matched_pass1 = pass1_pool[exact_mask].copy()
matched_pass1['match_score'] = 100
matched_pass1['match_type'] = 'Pass 1 - Exact'

# Deduplicate to enforce 1-to-1 mapping
matched_pass1 = matched_pass1.drop_duplicates(subset=['sofifa_uid']).drop_duplicates(subset=['sswy_uid'])

# Update remaining pools
remaining_sofifa = df_sofifa[~df_sofifa['sofifa_uid'].isin(matched_pass1['sofifa_uid'])]
remaining_sswy = df_sswy[~df_sswy['sswy_uid'].isin(matched_pass1['sswy_uid'])]


# --- PASS 2: Geographic Token-Set Fuzzy Match (>= 85%) ---
pass2_pool = pd.merge(
    remaining_sofifa, remaining_sswy, 
    left_on=['sofifa_country', 'sofifa_Season_Version'], 
    right_on=['soccersolver_country', 'soccersolver_season']
)

if not pass2_pool.empty:
    pass2_pool['match_score'] = pass2_pool.apply(calculate_fuzz, axis=1)
    matched_pass2 = pass2_pool[pass2_pool['match_score'] >= 85].copy()
    matched_pass2['match_type'] = 'Pass 2 - High Confidence Fuzzy'
    
    # Sort by score descending and deduplicate (keeps the highest score)
    matched_pass2 = matched_pass2.sort_values('match_score', ascending=False)
    matched_pass2 = matched_pass2.drop_duplicates(subset=['sofifa_uid']).drop_duplicates(subset=['sswy_uid'])
else:
    matched_pass2 = pd.DataFrame()

# Update remaining pools
remaining_sofifa = remaining_sofifa[~remaining_sofifa['sofifa_uid'].isin(matched_pass2['sofifa_uid'])]
remaining_sswy = remaining_sswy[~remaining_sswy['sswy_uid'].isin(matched_pass2['sswy_uid'])]


# --- PASS 3: Cross-Border Partial Match (65% to 84%) ---
# Block by Season ONLY (Drop Country)
pass3_pool = pd.merge(
    remaining_sofifa, remaining_sswy, 
    left_on=['sofifa_Season_Version'], 
    right_on=['soccersolver_season']
)

if not pass3_pool.empty:
    pass3_pool['match_score'] = pass3_pool.apply(calculate_fuzz, axis=1)
    partial_matches = pass3_pool[(pass3_pool['match_score'] >= 65) & (pass3_pool['match_score'] < 85)].copy()
    partial_matches['match_type'] = 'Pass 3 - Partial Match Review'
    
    partial_matches = partial_matches.sort_values('match_score', ascending=False)
    partial_matches = partial_matches.drop_duplicates(subset=['sofifa_uid']).drop_duplicates(subset=['sswy_uid'])
else:
    partial_matches = pd.DataFrame()

# Update unmatched sets
unmatched_sofifa = remaining_sofifa[~remaining_sofifa['sofifa_uid'].isin(partial_matches['sofifa_uid'])].copy()
unmatched_sswy = remaining_sswy[~remaining_sswy['sswy_uid'].isin(partial_matches['sswy_uid'])].copy()

# ==========================================
# 3. CONSOLIDATE AND EXPORT
# ==========================================
final_matched = pd.concat([matched_pass1, matched_pass2], ignore_index=True)

cols_to_drop = ['sofifa_uid', 'sswy_uid', 'sofifa_clean_name', 'sswy_clean_ss_name', 'sswy_clean_wy_name']

# Drop columns on the fly during export so they remain in memory for the diagnostic checks
final_matched.drop(columns=[c for c in cols_to_drop if c in final_matched.columns]).to_csv('newdata/teams/matched/MATCHED_100.csv', index=False)
partial_matches.drop(columns=[c for c in cols_to_drop if c in partial_matches.columns]).to_csv('newdata/teams/partial match/PARTIAL_MATCH_REVIEW.csv', index=False)
unmatched_sofifa.drop(columns=[c for c in cols_to_drop if c in unmatched_sofifa.columns]).to_csv('newdata/teams/no match/UNMATCHED_SOFIFA.csv', index=False)
unmatched_sswy.drop(columns=[c for c in cols_to_drop if c in unmatched_sswy.columns]).to_csv('newdata/teams/no match/UNMATCHED_SSWY.csv', index=False)
# =====================================================================
# --- JUPYTER-FRIENDLY VERIFICATION PROTOCOL ---
# =====================================================================

display(HTML("<h3>--- PHASE 2: PIPELINE DIAGNOSTIC REPORT ---</h3>"))

total_matched = len(final_matched)
total_partial = len(partial_matches)
total_un_sofifa = len(unmatched_sofifa)
total_un_sswy = len(unmatched_sswy)

# Check 1: No Sofifa Data Lost
sofifa_sum = total_matched + total_partial + total_un_sofifa
sofifa_integrity = (sofifa_sum == total_sofifa_initial)

# Check 2: No SSWY Data Lost
sswy_sum = total_matched + total_partial + total_un_sswy
sswy_integrity = (sswy_sum == total_sswy_initial)

# Check 3: Zero Cross-Contamination
final_matched_uids = set(matched_pass1['sofifa_uid']).union(set(matched_pass2['sofifa_uid']))
partial_uids = set(partial_matches['sofifa_uid'])
unmatched_uids = set(unmatched_sofifa['sofifa_uid'])

intersection_check = len(final_matched_uids.intersection(partial_uids)) == 0 and \
                     len(final_matched_uids.intersection(unmatched_uids)) == 0

# 1. Funnel Metrics
funnel_data = {
    'Pipeline Stage': ['Pass 1 (Exact)', 'Pass 2 (Fuzzy >85%)', 'Total 100% Matched', 'Pass 3 (Partial 65-84%)', 'Unmatched Sofifa', 'Unmatched SSWY'],
    'Row Count': [len(matched_pass1), len(matched_pass2), total_matched, total_partial, total_un_sofifa, total_un_sswy]
}
df_funnel = pd.DataFrame(funnel_data)

# 2. Integrity Checks
integrity_data = {
    'Test': ["Sofifa Conservation Check", "SSWY Conservation Check", "Cross-Contamination Check"],
    'Logic': [f"{sofifa_sum} Output == {total_sofifa_initial} Input", 
              f"{sswy_sum} Output == {total_sswy_initial} Input", 
              "Zero overlap between dataframes"],
    'Status': ['PASS' if sofifa_integrity else 'FAIL', 
               'PASS' if sswy_integrity else 'FAIL',
               'PASS' if intersection_check else 'FAIL']
}
df_integrity = pd.DataFrame(integrity_data)

# Display Tables
display(HTML("<b>1. Matching Funnel Results</b>"))
display(df_funnel.style.hide(axis="index"))

display(HTML("<br><b>2. Data Conservation & Integrity Checks</b>"))
display(df_integrity.style.hide(axis="index"))

Pipeline Stage,Row Count
Pass 1 (Exact),2249
Pass 2 (Fuzzy >85%),420
Total 100% Matched,2669
Pass 3 (Partial 65-84%),399
Unmatched Sofifa,767
Unmatched SSWY,4613


Test,Logic,Status
Sofifa Conservation Check,3835 Output == 3835 Input,PASS
SSWY Conservation Check,7681 Output == 7681 Input,PASS
Cross-Contamination Check,Zero overlap between dataframes,PASS


In [7]:
# 1. Load Data
df_partial = pd.read_csv('newdata/teams/partial match/PARTIAL_MATCH_REVIEW.csv')
initial_partials = len(df_partial)

# 2. Geographic Filter Logic
def check_geo(row):
    sf_c = str(row['sofifa_country']).strip().lower()
    ss_c = str(row['soccersolver_country']).strip().lower()
    # If either country is missing or marked unknown, keep it for manual review to be safe
    if pd.isna(row['sofifa_country']) or pd.isna(row['soccersolver_country']) or sf_c == 'unknown' or ss_c == 'unknown':
        return True
    return sf_c == ss_c

geo_pass_mask = df_partial.apply(check_geo, axis=1)

# 3. Youth Team Filter Logic
# Matches whole words for common reserve/youth identifiers
youth_pattern = r'\b(u16|u17|u18|u19|u20|u21|u22|u23|b|ii|castilla|reserves?)\b'

def is_youth_team(text):
    if pd.isna(text): return False
    return bool(re.search(youth_pattern, str(text).lower()))

def check_youth(row):
    ss_youth = is_youth_team(row['soccersolver_name'])
    wy_youth = is_youth_team(row['wyscout_name'])
    sf_youth = is_youth_team(row['sofifa_Team_Name'])
    
    # If target is a youth team but the Sofifa record is a first-team, reject it
    if (ss_youth or wy_youth) and not sf_youth:
        return False 
    return True

youth_pass_mask = df_partial.apply(check_youth, axis=1)

# 4. Apply Filters and Split Data
final_pass_mask = geo_pass_mask & youth_pass_mask

df_review = df_partial[final_pass_mask].copy()
df_auto_rejected = df_partial[~final_pass_mask].copy()

# 5. Insert Approval Column
df_review.insert(0, 'APPROVED', '')

# 6. Export
df_review.to_csv('newdata/teams/partial match/PARTIAL_MATCH_REVIEW_WITH_APPROVAL.csv', index=False)
df_auto_rejected.to_csv('newdata/teams/partial match/AUTO_REJECTED_PARTIALS.csv', index=False)


# =====================================================================
# --- STEP 3A VERIFICATION PROTOCOL ---
# =====================================================================
display(HTML("<h3>--- STEP 3A: AUTOMATED FILTERING DIAGNOSTIC ---</h3>"))

total_review = len(df_review)
total_rejected = len(df_auto_rejected)
integrity_pass = (total_review + total_rejected) == initial_partials

funnel_data = {
    'Category': ['Total Initial Partials', 'Auto-Rejected (Geo/Youth Mismatch)', 'Remaining for Manual Review'],
    'Count': [initial_partials, total_rejected, total_review]
}
df_funnel = pd.DataFrame(funnel_data)

integrity_data = {
    'Test': ["Mass Balance Conservation"],
    'Logic': [f"{total_review} (Review) + {total_rejected} (Rejected) == {initial_partials} (Initial)"],
    'Status': ['PASS' if integrity_pass else 'FAIL']
}
df_integrity = pd.DataFrame(integrity_data)

display(HTML("<b>1. Triage Funnel</b>"))
display(df_funnel.style.hide(axis="index"))
display(HTML("<br><b>2. Integrity Check</b>"))
display(df_integrity.style.hide(axis="index"))

Category,Count
Total Initial Partials,399
Auto-Rejected (Geo/Youth Mismatch),357
Remaining for Manual Review,42


Test,Logic,Status
Mass Balance Conservation,42 (Review) + 357 (Rejected) == 399 (Initial),PASS


In [9]:
# 1. Load All Active Files
df_matched = pd.read_csv('newdata/teams/matched/MATCHED_100.csv')
df_un_sof = pd.read_csv('newdata/teams/no match/UNMATCHED_SOFIFA.csv')
df_un_sswy = pd.read_csv('newdata/teams/no match/UNMATCHED_SSWY.csv')

df_auto_rej = pd.read_csv('newdata/teams/partial match/AUTO_REJECTED_PARTIALS.csv')
df_manual = pd.read_csv('newdata/teams/partial match/PARTIAL_MATCH_REVIEW_WITH_APPROVAL.csv')

initial_matched_count = len(df_matched)
initial_un_sof_count = len(df_un_sof)
initial_un_sswy_count = len(df_un_sswy)
initial_partial_count = len(df_auto_rej) + len(df_manual)

# Safely convert blanks to 0 and force all inputs (like 1.0 or '1') into flat integers
df_manual['APPROVED'] = pd.to_numeric(df_manual['APPROVED'], errors='coerce').fillna(0).astype(int)

approved_mask = df_manual['APPROVED'] == 1

# Drop the APPROVED column before merging so it doesn't pollute the final dataset
df_approved = df_manual[approved_mask].drop(columns=['APPROVED']).copy()
df_manual_rejected = df_manual[~approved_mask].drop(columns=['APPROVED']).copy()

# 3. Consolidate All Rejected Records
df_all_rejected = pd.concat([df_auto_rej, df_manual_rejected], ignore_index=True)

# 4. Generate Final Matched Dataset
FINAL_MATCHED = pd.concat([df_matched, df_approved], ignore_index=True)

# 5. Shatter Rejected Records & Append to Unmatched
# Dynamically separate the Sofifa and SSWY halves based on column prefixes
sofifa_cols = [c for c in df_all_rejected.columns if c.startswith('sofifa_')]
sswy_cols = [c for c in df_all_rejected.columns if not c.startswith('sofifa_') and c not in ['match_score', 'match_type', 'processed_at']]

rejected_sofifa = df_all_rejected[sofifa_cols].copy()
rejected_sswy = df_all_rejected[sswy_cols].copy()

FINAL_UNMATCHED_SOFIFA = pd.concat([df_un_sof, rejected_sofifa], ignore_index=True)
FINAL_UNMATCHED_SSWY = pd.concat([df_un_sswy, rejected_sswy], ignore_index=True)

# 6. Export the Master Files
FINAL_MATCHED.to_csv('newdata/teams/matched/FINAL_MATCHED_MASTER.csv', index=False)
FINAL_UNMATCHED_SOFIFA.to_csv('newdata/teams/no match/FINAL_UNMATCHED_SOFIFA_MASTER.csv', index=False)
FINAL_UNMATCHED_SSWY.to_csv('newdata/teams/no match/FINAL_UNMATCHED_SSWY_MASTER.csv', index=False)


# =====================================================================
# --- STEP 3B VERIFICATION PROTOCOL ---
# =====================================================================
display(HTML("<h3>--- STEP 3B: FINAL MASTER CONSOLIDATION REPORT ---</h3>"))

total_approved = len(df_approved)
total_rejected = len(df_all_rejected)

final_match_count = len(FINAL_MATCHED)
final_sof_count = len(FINAL_UNMATCHED_SOFIFA)
final_sswy_count = len(FINAL_UNMATCHED_SSWY)

# Constraint Checks
partial_split_pass = (total_approved + total_rejected) == initial_partial_count
match_sum_pass = final_match_count == (initial_matched_count + total_approved)
sof_sum_pass = final_sof_count == (initial_un_sof_count + total_rejected)
sswy_sum_pass = final_sswy_count == (initial_un_sswy_count + total_rejected)

report_data = {
    'Dataset': ['Final Matched Master', 'Final Unmatched Sofifa Master', 'Final Unmatched SSWY Master'],
    'Row Count': [final_match_count, final_sof_count, final_sswy_count],
    'Added from Partials': [f"+{total_approved}", f"+{total_rejected}", f"+{total_rejected}"]
}
df_report = pd.DataFrame(report_data)

integrity_data = {
    'Test': ["Partial Triage Conservation", "Matched Dataset Conservation", "Unmatched Sofifa Conservation", "Unmatched SSWY Conservation"],
    'Status': ['PASS' if partial_split_pass else 'FAIL', 
               'PASS' if match_sum_pass else 'FAIL',
               'PASS' if sof_sum_pass else 'FAIL',
               'PASS' if sswy_sum_pass else 'FAIL']
}
df_integrity = pd.DataFrame(integrity_data)

display(HTML("<b>1. Master File Outputs</b>"))
display(df_report.style.hide(axis="index"))
display(HTML("<br><b>2. Master Integrity Checks</b>"))
display(df_integrity.style.hide(axis="index"))

Dataset,Row Count,Added from Partials
Final Matched Master,2674,+5
Final Unmatched Sofifa Master,1161,+394
Final Unmatched SSWY Master,5007,+394


Test,Status
Partial Triage Conservation,PASS
Matched Dataset Conservation,PASS
Unmatched Sofifa Conservation,PASS
Unmatched SSWY Conservation,PASS
